# 🎬 03 — SVD Matrix Factorization
Implements hyperparameter tuning using GridSearchCV on a subset of the training set. Trains the final SVD model using the optimal configuration, and saves the weights as a pickle file.

In [1]:
import sys
import os
from pathlib import Path
sys.path.append(str(Path(os.getcwd()).parent))
import pandas as pd
from surprise.model_selection import GridSearchCV
from surprise import SVD
from src.models.svd_model import train_svd, save_model, build_surprise_dataset, load_model
from src.evaluation import compute_rmse

# Read splits
train_df = pd.read_parquet('../data/processed/train.parquet')
test_df = pd.read_parquet('../data/processed/test.parquet')

# Grid search on a sample to optimize training speed
gs_sample = train_df.sample(min(100000, len(train_df)), random_state=42)
data = build_surprise_dataset(gs_sample)

param_grid = {
    'n_factors': [50, 100],
    'n_epochs': [15, 20],
    'lr_all': [0.005, 0.007],
    'reg_all': [0.02, 0.05]
}

print("Running GridSearchCV for SVD...")
gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=2, n_jobs=-1)
gs.fit(data)

print(f"Best RMSE: {gs.best_score['rmse']:.4f}")
print(f"Best Params: {gs.best_params['rmse']}")

best_params = gs.best_params['rmse']

# Train final model on full trainset
final_svd = train_svd(train_df, **best_params)
save_model(final_svd, '../models/svd_model.pkl')

Running GridSearchCV for SVD...


Best RMSE: 1.0260
Best Params: {'n_factors': 50, 'n_epochs': 20, 'lr_all': 0.007, 'reg_all': 0.05}


Training SVD model (factors=50, epochs=20, lr=0.007, reg=0.05)...


Processing epoch 0


Processing epoch 1


...

Processing epoch 16


Processing epoch 17


Processing epoch 18


Processing epoch 19


SVD model successfully saved to ../models/svd_model.pkl
